# MolmoMotion × FiftyOne — PointMotionBench (real data, DAVIS-only start)

Language-guided **3D motion forecasting** explored in FiftyOne, on the real **PointMotionBench**
benchmark ([blog](https://allenai.org/blog/molmo-motion), [dataset](https://huggingface.co/datasets/allenai/PointMotionBench), [paper](https://arxiv.org/abs/2606.18558)).

This notebook loads genuine reconstructed clips with their per-object 3D + 2D tracked points and
human-verified captions, attaches **precomputed MolmoMotion predictions**, and gives you an
evaluation-and-failure-analysis workflow: sort by 3D displacement error, filter by motion type /
object / source split, and surface the worst cases.

It is configured **DAVIS-only** to start (the smallest, least-gated split). HOT3D and WorldTrack can
be enabled later in Section 1.

**No GPU is used in this notebook.** Model predictions are produced separately by the bundled
`run_molmomotion_inference.py` and loaded here as cached `.npz` files.

## ⬇️ Exact download URLs and commands (read first)

All data lives under a single configurable root — `DATA_ROOT` in Section 1, which defaults to a
`pointmotion_data` folder in your home directory. Adjust it to taste; every path below derives from
it. The commands assume your Python virtual environment (with FiftyOne 1.17 installed) is **already
activated**, so `hf`, `pip`, and `python` resolve to it.

> If the Hugging Face CLI is missing: `pip install -U "huggingface_hub[cli]"`. Newer versions expose
> the command as `hf`; older installs expose `huggingface-cli` — either works below.

**1 — PointMotionBench repo** (annotations, captions, indices, reconstruction scripts) → `$DATA_ROOT/PointMotionBench` (~3 GB)
```bash
HF_HUB_DISABLE_XET=1 hf download allenai/PointMotionBench \
    --repo-type dataset --local-dir "$DATA_ROOT/PointMotionBench"
```
`HF_HUB_DISABLE_XET=1` avoids the Xet stall the dataset card warns about; the download is resumable.
Page: `https://huggingface.co/datasets/allenai/PointMotionBench`

**2 — DAVIS 2017 source video** → `$DATA_ROOT/DAVIS` (~7.8 GB) — CC BY-NC 4.0
The only DAVIS file needed: **TrainVal 2017, Images + Annotations, 480p** (one zip, contains both).
```bash
mkdir -p "$DATA_ROOT/DAVIS" && cd "$DATA_ROOT/DAVIS"
curl -L -O https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip
unzip DAVIS-2017-trainval-480p.zip
ls            # NOTE: this usually extracts into a nested DAVIS/ folder (see below)
```
Terms page: `https://davischallenge.org/davis2017/code.html`
Do **not** download Full-Resolution, Unsupervised, Test-Dev, Test-Challenge, Semantics, or Scribbles.

> **DAVIS nested-folder note:** the zip typically extracts to `$DATA_ROOT/DAVIS/DAVIS/JPEGImages/480p`
> (a `DAVIS/` inside `DAVIS/`). After unzip, list the directory. If you see another `DAVIS` folder,
> either flatten it or leave it — the reconstruction step in Section 3 auto-detects the nested layout.
> It needs `JPEGImages/480p` and `Annotations/480p` under the DAVIS source root.

**3 — MolmoMotion weights** (only for *producing predictions* in the separate script — not by this notebook) → `$DATA_ROOT/models/...`
Two released checkpoints (both ~5B, Molmo2 backbone; suffixes = frame-history **H** / forecast-horizon **F**):
- `allenai/MolmoMotion-4B-H3-F30` — 3-frame history (the paper's stronger "3f" variant)
- `allenai/MolmoMotion-4B-H1-F32` — 1-frame history ("1f" variant)
```bash
hf download allenai/MolmoMotion-4B-H3-F30 --local-dir "$DATA_ROOT/models/MolmoMotion-4B-H3-F30"
# if it stalls:  HF_HUB_DISABLE_XET=1 hf download allenai/MolmoMotion-4B-H3-F30 --local-dir ...
```
Public download, no login. ~10 GB per checkpoint in bf16.
Collection: `https://huggingface.co/collections/allenai/molmomotion` · Code: `https://github.com/allenai/molmo-motion` · Backbone: `https://github.com/allenai/molmo2`

**Resulting `$DATA_ROOT` layout:**
```
$DATA_ROOT/
├── PointMotionBench/              # step 1  (reconstruct writes davis/videos/ in here)
├── DAVIS/                         # step 2  (JPEGImages/480p + Annotations/480p)
├── models/MolmoMotion-4B-H3-F30/  # step 3  (only needed for the inference script)
└── molmomotion_predictions/       # created by run_molmomotion_inference.py --out
```

**Licenses you accept via the flags in Section 1:** DAVIS = CC BY-NC 4.0 (non-commercial + attribution);
PointMotionBench = Ai2 Responsible Use. HOT3D (when enabled) is gated and needs `hf auth login`.

---
### Download troubleshooting (common snags)

- **`PermissionError: [Errno 1] Operation not permitted` mentioning `os.getcwd()`** — your shell's
  current directory was moved/deleted out from under it (e.g. you flattened the DAVIS folder while
  standing inside it). Fix: `cd` to a valid directory (such as your home directory) and retry. The
  download's `--local-dir` is absolute, so it runs fine from anywhere.
- **`httpx.ConnectTimeout: handshake operation timed out`** — a transient network/SSL stall, not your
  setup. The download is **resumable**: re-run the same command and it skips what's already on disk.
  For flaky connections, loop it so it auto-resumes:
  ```bash
  until HF_HUB_DISABLE_XET=1 hf download allenai/PointMotionBench \
          --repo-type dataset --local-dir "$DATA_ROOT/PointMotionBench"; do
    echo "timed out, resuming in 5s..."; sleep 5
  done
  ```
  Lengthening timeouts and disabling the parallel accelerator also helps:
  `export HF_HUB_DOWNLOAD_TIMEOUT=60 HF_HUB_ETAG_TIMEOUT=60` and prefix `HF_HUB_ENABLE_HF_TRANSFER=0`.
  On VPN/corporate Wi-Fi, dropping it often clears handshake timeouts instantly.
- **Verify when done:** check the repo size is ~3 GB and that it contains the annotation files plus
  `davis/ hot3d/ worldtrack/` script folders.

### Assumptions
- Running inside a **Python virtual environment with FiftyOne 1.17 installed and working** (this
  notebook does not create or manage the environment).
- `huggingface_hub`, `numpy`, `imageio[ffmpeg]`, `opencv-python-headless`, and `ffmpeg` (which
  provides `ffprobe`) available. The setup cell below checks for these and installs the missing
  Python ones; `ffmpeg` is a system package (e.g. `brew install ffmpeg` / `apt install ffmpeg`).

In [ ]:
# --- Environment check (assumes a venv with FiftyOne 1.17 is already active) ---
import importlib, subprocess, sys, shutil

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        m = importlib.import_module(name)
        return getattr(m, "__version__", "ok")
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return getattr(importlib.import_module(name), "__version__", "ok")

import fiftyone as fo
print("fiftyone:", fo.__version__)
for p, n in [("numpy", None), ("huggingface_hub", None),
             ("imageio", None), ("imageio-ffmpeg", "imageio_ffmpeg"),
             ("opencv-python-headless", "cv2")]:
    print(f"{p}:", ensure(p, n))
_ff = shutil.which("ffmpeg") is not None and shutil.which("ffprobe") is not None
print("ffmpeg + ffprobe on PATH:", _ff)
if not _ff:
    print("  -> install the system package (e.g. `brew install ffmpeg` or `apt install ffmpeg`);"
          " ffprobe is needed for video metadata. The dataset still builds without it,"
          " but you'll see metadata warnings.")

## 1 · Configuration — paths and license acceptance

Set `DATA_ROOT` to wherever you downloaded the data; every other path derives from it. **Accept each
license** by setting its flag to `True` — this is your explicit consent record. DAVIS and Ai2 are
pre-set to `True`; flip them back to `False` if you do not agree (DAVIS is non-commercial only).

In [ ]:
from pathlib import Path

# ---- Single data root; everything else derives from it -------------------------
# Override with an env var if you like:  export POINTMOTION_DATA_ROOT=/path/to/data
import os
DATA_ROOT = Path(os.environ.get("POINTMOTION_DATA_ROOT", "~/pointmotion_data")).expanduser()

POINTMOTIONBENCH_ROOT = DATA_ROOT / "PointMotionBench"

# Reconstructed *videos* per source (outputs of the Ai2 reconstruct_*.py scripts).
DAVIS_VIDEO_DIR      = POINTMOTIONBENCH_ROOT / "davis"      / "videos" / "input_480p"
HOT3D_VIDEO_DIR      = POINTMOTIONBENCH_ROOT / "hot3d"      / "rgbs"
WORLDTRACK_VIDEO_DIR = POINTMOTIONBENCH_ROOT / "worldtrack"

# Source data for reconstruction.
DAVIS_SOURCE      = DATA_ROOT / "DAVIS"             # DAVIS-2017-trainval-480p (see nested-folder note)
HOT3D_WORKDIR     = DATA_ROOT / "hot3d_work"
WORLDTRACK_SOURCE = DATA_ROOT / "WorldTrack"

# Precomputed MolmoMotion predictions (output of run_molmomotion_inference.py --out).
PREDICTIONS_DIR = DATA_ROOT / "molmomotion_predictions"

# ---- Which source splits to load (DAVIS-only to start) ----
USE_DAVIS      = True
USE_HOT3D      = False   # enable later; HOT3D is gated (needs hf auth login)
USE_WORLDTRACK = False   # enable later

# ---- License acceptance (set True only if you actually agree) ----
ACCEPT_DAVIS_CC_BY_NC_4_0   = True    # CC BY-NC 4.0 — https://davischallenge.org/davis2017/code.html
ACCEPT_HOT3D_ARIA_TERMS     = False   # https://huggingface.co/datasets/bop-benchmark/hot3d  + Project Aria
ACCEPT_WORLDTRACK_TERMS     = False   # https://github.com/HavenFeng/St4RTrack
ACCEPT_AI2_RESPONSIBLE_USE  = True    # https://allenai.org/responsible-use

DATASET_NAME = "pointmotionbench-real"
N_QUERY = 8   # MolmoMotion uses 8 query points per object

print("DATA_ROOT:", DATA_ROOT)
print("Splits:", dict(DAVIS=USE_DAVIS, HOT3D=USE_HOT3D, WorldTrack=USE_WORLDTRACK))
print("Acceptance:",
      dict(DAVIS=ACCEPT_DAVIS_CC_BY_NC_4_0, HOT3D=ACCEPT_HOT3D_ARIA_TERMS,
           WorldTrack=ACCEPT_WORLDTRACK_TERMS, Ai2=ACCEPT_AI2_RESPONSIBLE_USE))

## 2 · Preflight gate

Verifies, for each enabled split, that (a) you accepted the license and (b) the reconstructed videos
are on disk — plus the repo and predictions. Raises with a precise checklist until everything is
ready. Nothing downstream runs until this passes.

Set `PREDICTIONS_OPTIONAL = True` to load real clips + ground truth *without* predictions yet (the
`ade_*` columns populate later, once you add the `.npz` files).

In [ ]:
def _count_videos(d, exts=(".mp4", ".npz")):
    p = Path(d)
    return 0 if not p.exists() else sum(1 for f in p.rglob("*") if f.suffix.lower() in exts)

class PreflightError(RuntimeError):
    pass

def preflight():
    problems, ready = [], {}

    if not ACCEPT_AI2_RESPONSIBLE_USE:
        problems.append("Set ACCEPT_AI2_RESPONSIBLE_USE = True "
                        "(https://allenai.org/responsible-use) to use PointMotionBench.")
    if not POINTMOTIONBENCH_ROOT.exists():
        problems.append(f"PointMotionBench repo not found at {POINTMOTIONBENCH_ROOT}. "
                        f"Run Section 3, Step A (hf download allenai/PointMotionBench).")

    checks = [
        ("DAVIS",      USE_DAVIS,      ACCEPT_DAVIS_CC_BY_NC_4_0,
         "ACCEPT_DAVIS_CC_BY_NC_4_0", DAVIS_VIDEO_DIR, "Step B"),
        ("HOT3D",      USE_HOT3D,      ACCEPT_HOT3D_ARIA_TERMS,
         "ACCEPT_HOT3D_ARIA_TERMS", HOT3D_VIDEO_DIR, "Step C"),
        ("WorldTrack", USE_WORLDTRACK, ACCEPT_WORLDTRACK_TERMS,
         "ACCEPT_WORLDTRACK_TERMS", WORLDTRACK_VIDEO_DIR, "Step D"),
    ]
    for name, enabled, accepted, flag, vdir, step in checks:
        if not enabled:
            continue
        if not accepted:
            problems.append(f"{name}: enabled but terms not accepted. Set {flag} = True ({step}).")
            continue
        n = _count_videos(vdir)
        if n == 0:
            problems.append(f"{name}: terms accepted but no reconstructed clips in {vdir}. "
                            f"Run Section 3, {step}.")
        else:
            ready[name] = n

    if _count_videos(PREDICTIONS_DIR, exts=(".npz", ".json")) == 0:
        problems.append(f"No precomputed predictions in {PREDICTIONS_DIR}. Produce them with "
                        f"run_molmomotion_inference.py (Section 5), then re-run this gate. "
                        f"(Tip: set PREDICTIONS_OPTIONAL = True to load clips without predictions.)")

    if problems:
        raise PreflightError("PREFLIGHT FAILED — resolve these before continuing:\n  - "
                             + "\n  - ".join(problems))
    print("PREFLIGHT PASSED. Splits ready:", ready)
    return ready

# Allow the dataset to load with reconstructed clips + GT but no predictions yet.
PREDICTIONS_OPTIONAL = False

def preflight_lenient():
    """Like preflight() but downgrades the missing-predictions error to a warning."""
    try:
        return preflight()
    except PreflightError as e:
        msg = str(e)
        only_preds = ("No precomputed predictions" in msg and msg.count("- ") == 1)
        if PREDICTIONS_OPTIONAL and only_preds:
            print("PREFLIGHT PASSED (predictions optional — none found yet, clips will load without them).")
            return {k: _count_videos(v) for k, v, en in
                    [("DAVIS", DAVIS_VIDEO_DIR, USE_DAVIS),
                     ("HOT3D", HOT3D_VIDEO_DIR, USE_HOT3D),
                     ("WorldTrack", WORLDTRACK_VIDEO_DIR, USE_WORLDTRACK)] if en and _count_videos(v)}
        raise

READY_SPLITS = preflight_lenient()

## 3 · Download & reconstruct (skip if preflight already passed)

Run only the steps for splits you enabled. Each is guarded by its acceptance flag. These mirror the
PointMotionBench dataset-card Setup steps. After running, re-run Section 2.

In [ ]:
import subprocess, os, shlex

def run(cmd, **env):
    e = os.environ.copy(); e.update({k: str(v) for k, v in env.items()})
    print("$", cmd)
    subprocess.check_call(cmd, shell=True, env=e)

# ---- Step A — PointMotionBench repo (annotations + scripts) ----
def step_A_download_repo():
    if not ACCEPT_AI2_RESPONSIBLE_USE:
        raise PreflightError("Set ACCEPT_AI2_RESPONSIBLE_USE = True first.")
    POINTMOTIONBENCH_ROOT.mkdir(parents=True, exist_ok=True)
    run(f"hf download allenai/PointMotionBench --repo-type dataset "
        f"--local-dir {shlex.quote(str(POINTMOTIONBENCH_ROOT))}",
        HF_HUB_DISABLE_XET="1")

# step_A_download_repo()

In [ ]:
# ---- Step B — DAVIS reconstruct mp4s (CC BY-NC 4.0) ----
# Requires DAVIS-2017-trainval-480p extracted into DAVIS_SOURCE (see nested-folder note up top).
def step_B_davis():
    if not ACCEPT_DAVIS_CC_BY_NC_4_0:
        raise PreflightError("DAVIS is CC BY-NC 4.0. Set ACCEPT_DAVIS_CC_BY_NC_4_0 = True after "
                             "reviewing https://davischallenge.org/davis2017/code.html")
    root = DAVIS_SOURCE
    if (root / "DAVIS" / "JPEGImages").exists() and not (root / "JPEGImages").exists():
        root = root / "DAVIS"
        print(f"(using nested DAVIS source: {root})")
    if not (root / "JPEGImages").exists():
        raise PreflightError(f"DAVIS source not found under {DAVIS_SOURCE} "
                             f"(expected JPEGImages/480p + Annotations/480p). See the nested-folder note.")
    DAVIS_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    run(f"python {shlex.quote(str(POINTMOTIONBENCH_ROOT / 'davis' / 'reconstruct_davis.py'))} "
        f"--davis-root {shlex.quote(str(root))} "
        f"--output-dir {shlex.quote(str(DAVIS_VIDEO_DIR))}")

# step_B_davis()

In [ ]:
# ---- Step C — HOT3D (Aria / BOP terms; gated) ----  [enable USE_HOT3D first]
def step_C_hot3d():
    if not ACCEPT_HOT3D_ARIA_TERMS:
        raise PreflightError("HOT3D is gated. Accept terms at "
                             "https://huggingface.co/datasets/bop-benchmark/hot3d (+ Project Aria), "
                             "run `hf auth login`, then set ACCEPT_HOT3D_ARIA_TERMS = True.")
    HOT3D_WORKDIR.mkdir(parents=True, exist_ok=True)
    HOT3D_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    run(f"python {shlex.quote(str(POINTMOTIONBENCH_ROOT / 'hot3d' / 'reconstruct_hot3d.py'))} "
        f"--workdir {shlex.quote(str(HOT3D_WORKDIR))} "
        f"--output-dir {shlex.quote(str(HOT3D_VIDEO_DIR))}")

# step_C_hot3d()

In [ ]:
# ---- Step D — WorldTrack (St4RTrack terms) ----  [enable USE_WORLDTRACK first]
def step_D_worldtrack():
    if not ACCEPT_WORLDTRACK_TERMS:
        raise PreflightError("Set ACCEPT_WORLDTRACK_TERMS = True after reviewing "
                             "https://github.com/HavenFeng/St4RTrack")
    if not WORLDTRACK_SOURCE.exists():
        raise PreflightError(f"WorldTrack source not found at {WORLDTRACK_SOURCE}.")
    WORLDTRACK_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    idx = POINTMOTIONBENCH_ROOT / "worldtrack" / "worldtrack_index_map.json"
    run(f"python {shlex.quote(str(POINTMOTIONBENCH_ROOT / 'worldtrack' / 'reconstruct_worldtrack.py'))} "
        f"--index_map {shlex.quote(str(idx))} "
        f"--src_dir {shlex.quote(str(WORLDTRACK_SOURCE))} "
        f"--output_dir {shlex.quote(str(WORLDTRACK_VIDEO_DIR))}")

# step_D_worldtrack()

# After running the steps you enabled, re-run Section 2 (preflight).

## 4 · Load the real annotations, captions, and tracks

Reads the actual PointMotionBench DAVIS layout (confirmed against the released repo):
- `davis/davis_captions.json` — `{clip: {"description": ...}}`, keyed by DAVIS sequence name.
- `davis/tracks/{clip}_3d.npz` — `points_3d` → `{object_name: (frames, points, 3)}`.
- `davis/tracks/{clip}_2d.npz` — `tracks` → `{object_name: (points, frames, 2)}`, plus `dim = [H, W]`.
- reconstructed `davis/videos/input_480p/{clip}.mp4` (from Section 3, Step B).

The loader unwraps the object-keyed dicts, takes the primary object's 3D track as ground truth,
transposes + normalizes the 2D track to `[0,1]` for keypoint overlay, subsamples to `N_QUERY` points,
and uses the object name as `category`. Add HOT3D/WorldTrack loaders to `SPLIT_PATHS` when you enable
those splits.

In [ ]:
import json, numpy as np

N_QUERY_KEEP = N_QUERY   # how many query points to keep (paper uses 8); None keeps all

SPLIT_PATHS = {
    "DAVIS": dict(
        captions = POINTMOTIONBENCH_ROOT / "davis" / "davis_captions.json",
        tracks   = POINTMOTIONBENCH_ROOT / "davis" / "tracks",
        videos   = DAVIS_VIDEO_DIR,
    ),
    # HOT3D / WorldTrack loaders can be added here when those splits are enabled.
}

def _unwrap(npz, key):
    "Return npz[key] unwrapped if it's a 0-d object array, else the array itself."
    if key not in npz.files:
        return None
    a = npz[key]
    return a.item() if a.shape == () else a

def _first_object(d):
    "Tracks dicts are keyed by object name; return (name, array) for the primary object."
    if not isinstance(d, dict) or not d:
        return None, None
    name = next(iter(d))
    return name, np.asarray(d[name], dtype=np.float32)

def _subsample_points(n_total, k):
    if k is None or k >= n_total:
        return np.arange(n_total)
    return np.linspace(0, n_total - 1, k).round().astype(int)

def load_davis(paths):
    captions = json.loads(Path(paths["captions"]).read_text())
    tracks_dir = Path(paths["tracks"])
    videos_dir = Path(paths["videos"])
    entries, skipped = [], 0

    for f3 in sorted(tracks_dir.glob("*_3d.npz")):
        clip = f3.name[:-len("_3d.npz")]
        vpath = videos_dir / f"{clip}.mp4"
        if not vpath.exists():
            skipped += 1
            continue

        d3 = np.load(f3, allow_pickle=True)
        obj3d = _unwrap(d3, "points_3d")
        obj_name, p3 = _first_object(obj3d)          # p3: (F, N, 3)
        if p3 is None:
            skipped += 1
            continue

        f2 = tracks_dir / f"{clip}_2d.npz"
        uv = None
        if f2.exists():
            d2 = np.load(f2, allow_pickle=True)
            t2 = _unwrap(d2, "tracks")
            dim = d2["dim"] if "dim" in d2.files else None     # [H, W]
            _, p2 = _first_object(t2)                          # (N, F, 2)
            if p2 is not None:
                p2 = np.transpose(p2, (1, 0, 2))               # -> (F, N, 2)
                if dim is not None:
                    H, W = float(dim[0]), float(dim[1])
                    p2 = p2 / np.array([W, H], dtype=np.float32)  # x/W, y/H -> [0,1]
                uv = p2

        N_ = p3.shape[1]
        idx = _subsample_points(N_, N_QUERY_KEEP)
        gt3d = p3[:, idx, :]                                   # (F, k, 3)
        uv_k = uv[:, idx, :] if uv is not None else None       # (F, k, 2)

        cap = captions.get(clip, {})
        caption = cap.get("description", "") if isinstance(cap, dict) else str(cap)

        entries.append(dict(
            clip_id=clip, split="DAVIS", video_path=str(vpath),
            caption=caption, motion_type="unknown",
            category=obj_name or "unknown",
            gt_3d=gt3d.astype(np.float32),
            gt_2d=uv_k.astype(np.float32) if uv_k is not None else None,
        ))
    return entries, skipped

def load_annotations():
    enabled = {"DAVIS": USE_DAVIS, "HOT3D": USE_HOT3D, "WorldTrack": USE_WORLDTRACK}
    all_entries, total_skip = [], 0
    for split, on in enabled.items():
        if not on:
            continue
        if split not in SPLIT_PATHS:
            print(f"  ! {split} enabled but no loader wired yet — skipping (add it to SPLIT_PATHS).")
            continue
        if split == "DAVIS":
            ents, sk = load_davis(SPLIT_PATHS["DAVIS"])
        else:
            ents, sk = [], 0
        print(f"{split}: loaded {len(ents)} clips (skipped {sk} with no matching video)")
        all_entries += ents
        total_skip += sk
    if not all_entries:
        raise PreflightError("No clips loaded. Check that videos are reconstructed (Section 3, "
                             "Step B) and that tracks/*.npz + captions exist under "
                             f"{POINTMOTIONBENCH_ROOT}.")
    print(f"TOTAL: {len(all_entries)} clips, GT 3D shape e.g. {all_entries[0]['gt_3d'].shape} "
          f"(frames, {N_QUERY_KEEP} query pts, xyz)")
    return all_entries

ENTRIES = load_annotations()

## 5 · Attach precomputed MolmoMotion predictions

**No GPU here.** Produce predictions on a GPU machine with the bundled `run_molmomotion_inference.py`,
then copy the `.npz` files into `PREDICTIONS_DIR`. One file per clip, `{clip_id}.npz`, with:

- `ar` — `(T, N_QUERY, 3)`: MolmoMotion-AR predicted 3D trajectory (meters, world frame)
- `fm` — `(T, N_QUERY, 3)`: MolmoMotion-FM predicted 3D trajectory *(optional)*

Example run (DAVIS, on the GPU box):
```bash
python run_molmomotion_inference.py \
    --pmb-root      "$DATA_ROOT/PointMotionBench" \
    --davis-videos  "$DATA_ROOT/PointMotionBench/davis/videos/input_480p" \
    --weights       allenai/MolmoMotion-4B-H3-F30 \
    --variant       ar \
    --splits        DAVIS \
    --out           "$DATA_ROOT/molmomotion_predictions"
```
The loader computes **3D Average Displacement Error** vs each clip's GT track. Clips without a
prediction are kept and flagged (`has_pred = False`).

In [ ]:
def _load_pred(clip_id):
    npz = PREDICTIONS_DIR / f"{clip_id}.npz"
    js  = PREDICTIONS_DIR / f"{clip_id}.json"
    if npz.exists():
        d = np.load(npz, allow_pickle=True)
        return (d["ar"].astype(np.float32) if "ar" in d.files else None,
                d["fm"].astype(np.float32) if "fm" in d.files else None)
    if js.exists():
        d = json.loads(js.read_text())
        f = lambda k: np.asarray(d[k], dtype=np.float32) if d.get(k) is not None else None
        return f("ar"), f("fm")
    return None, None

def ade_3d(pred, gt):
    "3D Average Displacement Error in meters."
    if pred is None or gt is None: return None
    n = min(pred.shape[0], gt.shape[0])
    return float(np.linalg.norm(pred[:n] - gt[:n], axis=-1).mean())

n_with_pred = 0
for e in ENTRIES:
    ar, fm = _load_pred(e["clip_id"])
    e["ar"], e["fm"] = ar, fm
    e["ade_ar"] = ade_3d(ar, e["gt_3d"])
    e["ade_fm"] = ade_3d(fm, e["gt_3d"])
    e["has_pred"] = ar is not None or fm is not None
    n_with_pred += int(e["has_pred"])
print(f"{n_with_pred}/{len(ENTRIES)} clips have predictions attached")

## 6 · Build the FiftyOne dataset (per-frame keypoints)

Each clip becomes a video `Sample`. The query points are attached **per frame**
(`frames[t]["gt_points"]`), so they move *with* the object as you scrub the video, instead of
painting the whole trajectory at once. Frames are 1-indexed in FiftyOne. The 2D paths are
subsampled to `N_QUERY` points and were normalized to `[0,1]` in Section 4. The raw 3D GT trajectory
stays on the sample for any 3D analysis. If you later project your MolmoMotion 3D predictions to 2D,
drop them into the entries as `pred_ar_2d` / `pred_fm_2d` `(F, N, 2)` and they'll render as per-frame
`pred_ar_points` / `pred_fm_points` too.

The launch cell sets a color scheme so ground-truth points render in red (and predicted points, when
present, in cyan).

In [ ]:
import fiftyone as fo

def add_per_frame_keypoints(sample, uv, label, n_query=N_QUERY):
    """uv: (F, N, 2) normalized in [0,1]. Attaches one Keypoints field per frame."""
    if uv is None:
        return 0
    uv = np.asarray(uv, dtype=np.float32)
    F_, N_ = uv.shape[0], uv.shape[1]
    k = min(n_query, N_)
    for t in range(F_):
        kps = [fo.Keypoint(label=f"{label}_p{j}", points=[[float(uv[t, j, 0]),
                                                            float(uv[t, j, 1])]])
               for j in range(k)]
        sample.frames[t + 1][label] = fo.Keypoints(keypoints=kps)   # 1-indexed
    return F_

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
ds = fo.Dataset(DATASET_NAME, persistent=True)

n_with_kp = 0
for e in ENTRIES:
    s = fo.Sample(filepath=e["video_path"])
    s["clip_id"]     = e["clip_id"]
    s["split"]       = e["split"]
    s["motion_type"] = e["motion_type"]
    s["category"]    = e["category"]
    s["caption"]     = e["caption"]
    s["has_pred"]    = e["has_pred"]
    if e["ade_ar"] is not None: s["ade_ar"] = e["ade_ar"]
    if e["ade_fm"] is not None: s["ade_fm"] = e["ade_fm"]
    cand = [(v, k) for k, v in [("AR", e["ade_ar"]), ("FM", e["ade_fm"])] if v is not None]
    if cand:
        s["ade_best"]   = min(c[0] for c in cand)
        s["best_model"] = min(cand)[1]

    if e.get("gt_2d") is not None:
        if add_per_frame_keypoints(s, e["gt_2d"], "gt_points", N_QUERY):
            n_with_kp += 1
    if e.get("pred_ar_2d") is not None:
        add_per_frame_keypoints(s, e["pred_ar_2d"], "pred_ar_points", N_QUERY)
    if e.get("pred_fm_2d") is not None:
        add_per_frame_keypoints(s, e["pred_fm_2d"], "pred_fm_points", N_QUERY)

    if e["gt_3d"] is not None:
        s["gt_traj_3d"] = e["gt_3d"].reshape(-1).tolist()
        s["traj_shape"] = list(e["gt_3d"].shape)

    ds.add_sample(s)

ds.compute_metadata()
print(f"built {len(ds)} samples; {n_with_kp} with per-frame gt_points "
      f"({N_QUERY} query pts/frame)")
print(ds)

## 7 · Evaluation views — sort, filter, surface failures

In [ ]:
from fiftyone import ViewField as F

have_ar = ds.exists("ade_ar")
have_fm = ds.exists("ade_fm")

print("=== Overall 3D ADE (meters, lower is better) ===")
if len(have_ar): print(f"MolmoMotion-AR : {have_ar.mean('ade_ar'):.3f}  (n={len(have_ar)})")
if len(have_fm): print(f"MolmoMotion-FM : {have_fm.mean('ade_fm'):.3f}  (n={len(have_fm)})")
if not len(have_ar) and not len(have_fm):
    print("(no predictions yet — add .npz files to PREDICTIONS_DIR and re-run Sections 5-7)")

print("\n=== By source split (AR) ===")
for split in ["DAVIS", "HOT3D", "WorldTrack"]:
    v = ds.match(F("split") == split).exists("ade_ar")
    if len(v): print(f"{split:11s} n={len(v):3d}  ADE_AR={v.mean('ade_ar'):.3f}")

print("\n=== Mean AR error by motion type (top 10) ===")
counts = ds.count_values("motion_type")
rows = []
for m in counts:
    v = ds.match(F("motion_type") == m).exists("ade_ar")
    if len(v): rows.append((m, counts[m], v.mean("ade_ar")))
for m, n, e in sorted(rows, key=lambda r: r[2], reverse=True)[:10]:
    print(f"{m:14s} n={n:3d}  ADE_AR={e:.3f}")

if len(have_ar):
    ds.save_view("worst_AR_predictions", ds.exists("ade_ar").sort_by("ade_ar", reverse=True))
ds.save_view("missing_predictions", ds.match(F("has_pred") == False))
for split in ["DAVIS", "HOT3D", "WorldTrack"]:
    v = ds.match(F("split") == split)
    if len(v): ds.save_view(f"split_{split}", v)
print("\nSaved views:", ds.list_saved_views())

if len(have_ar):
    print("\nTop-5 worst AR clips:")
    for s in ds.exists("ade_ar").sort_by("ade_ar", reverse=True)[:5]:
        print(f"  {str(s.caption)[:30]:30s} {s.split:10s} ADE_AR={s.ade_ar:.3f}")

## 8 · Launch the App

Sidebar filters: `split`, `motion_type`, `category`, `best_model`, `has_pred`, and the `ade_*`
sliders. The `gt_points` keypoints are **per-frame** — open a sample and scrub the video to watch
the query points track the object. Ground-truth points render in red; predicted points (when added)
in cyan. Saved views are in the bookmark menu.

In [ ]:
# Ground-truth points in red; predicted points (when present) in cyan.
color_scheme = fo.ColorScheme(fields=[
    {"path": "frames.gt_points",      "color": "#FF3333"},
    {"path": "frames.pred_ar_points", "color": "#00E5FF"},
    {"path": "frames.pred_fm_points", "color": "#FFB300"},
])

session = fo.launch_app(ds, color_scheme=color_scheme)
# In a notebook: session.show()
session

In [ ]:
# Optional cleanup
# fo.delete_dataset(DATASET_NAME)